# Importation et définition de variables globales

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import random
import ale_py
import gc
import seaborn as sns
import pickle

from agent.rainbow_agent import RainbowAgent

from agent.no_noisy_no_categ_agent import NoNoisyNoCategAgent
from reseaux.nn_no_noisy_no_categ_no_duel import Network as NnNoThree
from reseaux.nn_no_noisy_no_categ import Network as NnDueling

from agent.no_noisy_agent import NoNoisyAgent 
from reseaux.nn_no_noisy_no_duel import Network as NnCateg
from reseaux.nn_no_noisy import Network as NnNoNoisy

In [ ]:
# Environment
env = gym.make("ALE/Freeway-v5", render_mode="rgb_array")

# Set random seed

In [ ]:
seed = 777

def seed_torch(seed):
    torch.manual_seed(seed)
    if torch.backends.cudnn.enabled:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

np.random.seed(seed)
random.seed(seed)
seed_torch(seed)

# Initialisation

In [ ]:
# paramètres
num_frames = 100000
ss_num_frames = 5000
memory_size = 100000 
batch_size = 128
target_update = 100
epsilon_decay = 1 / 2000
buffer_dir = './buffer_saves_parallel'
# Par défait reward_function est à 2
reward_function = 1 # 1 = reward_V1, 2 = reward_V2, 3 = reward simpliste

In [ ]:
videos_path = ["videos/videos_rainbow/", "videos/videos_no_three/","videos/videos_no_noisy_no_categ/","videos/videos_no_noisy_no_duel/", "videos/videos_no_noisy/"]

# Définition de l'agent

In [ ]:
score_agents = np.zeros([1, num_frames//ss_num_frames], dtype=float)
nb_crashes_agents = np.zeros([1, num_frames//ss_num_frames], dtype=int)
# Définition
agent = RainbowAgent(env, memory_size, batch_size, target_update, seed, seed, f'{buffer_dir}/rainbow', reward_function)
nom_agent = "rainbow"
#agent = NoNoisyNoCategAgent(NnNoThree, env, memory_size, batch_size, target_update, epsilon_decay, seed, seed, f'{buffer_dir}/no_three', reward_function)
#nom_agent = "no_three"
#agent = NoNoisyNoCategAgent(NnDueling, env, memory_size, batch_size, target_update, epsilon_decay, seed, seed, f'{buffer_dir}/no_noisy_no_categ', reward_function)
#nom_agent = "no_noisy_no_categ"
#agent = NoNoisyAgent(NnCateg, env, memory_size, batch_size, target_update,  epsilon_decay, seed, seed, f'{buffer_dir}/no_noisy_no_duel', reward_function)
#nom_agent = "no_noisy_no_duel"
#agent = NoNoisyAgent(NnNoNoisy, env, memory_size, batch_size, target_update, epsilon_decay, seed, seed, f'{buffer_dir}/no_noisy', reward_function)
#nom_agent = "no_noisy"

# Entraînement et test par intervalle de ss_num_frames

In [ ]:
for iter in range(0, num_frames, ss_num_frames): 
   
   # Entraînement
   agent.train(iter)

   # Test
   score_agents[0][iter//ss_num_frames], nb_crashes_agents[0][iter//ss_num_frames] = agent.test("corbeille/" + nom_agent + "/")

   #Sauvegarde préventive en cas d'interruption
   if iter % ss_num_frames == 0:
      # Sauvegarde
      with open("score_agents_" + nom_agent + "_" + str(num_frames) + ".pkl", "wb") as f:
         pickle.dump(score_agents, f)
      with open("nb_crashes_" + nom_agent + "_" + str(num_frames) + ".pkl", "wb") as g:
         pickle.dump(nb_crashes_agents, g)
      print(f"Liste sauvegardée : {iter}")

In [ ]:
# Enregistrer les fichiers localement
with open("score_agents_" + nom_agent + "_" + str(num_frames) + ".pkl", "rb") as f:
    score_agents = pickle.load(f)
         
with open("nb_crashes_" + nom_agent + "_" + str(num_frames) + ".pkl", "rb") as g:
    nb_crashes_agents = pickle.load(g)

print("Liste chargée :", type(score_agents))  # Vérifie le type de l'objet
print("Liste chargée :", type(nb_crashes_agents))  # Vérifie le type de l'objet

On affiche le contenu des listes :

In [ ]:
score_agents

In [ ]:
nb_crashes_agents